In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f


Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [86]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})


KeyboardInterrupt: 

In [4]:
path_to_release_folder = "../../data/25.06/"
path_to_intermediate_data_folder = "../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)

all_evidence = session.spark.read.parquet(path_to_release_folder + "output/evidence")


efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

chembl_evidence.show(1)

l2g_full = session.spark.read.parquet(path_to_intermediate_data_folder + "l2g_full_for_enrichment/")
l2g_full.count()

l2g_full.show(1)


g_p_s = session.spark.read.parquet(path_to_intermediate_data_folder + "genes_therapeutic_areas")
g_p_s.count()


26/01/21 14:06:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+----------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+-----------------+-----------------------

8285

In [ ]:
columns_to_select = ["targetId", "diseaseId"]


# L2G evidence


In [ ]:
l2g_full_diseases = l2g_full.drop("diseaseIds")


In [ ]:
l2g_full_diseases.count()


70400

In [ ]:
l2g_evidence = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full_diseases,
    score_column="score",
    datasource_id="l2g_combined",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)


In [ ]:
l2g_evidence_diseases = (
    l2g_evidence.select(columns_to_select)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("all_diseases"))
    .cache()
)
l2g_evidence_diseases.count()


36858

In [ ]:
l2g_evidence_VEP = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full_diseases.filter(f.col("VEP") == 1),
    score_column="score",
    datasource_id="l2g_VEP",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)


In [ ]:
l2g_evidence_VEP = (
    l2g_evidence_VEP.select(columns_to_select)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("gwas_with_pav"))
    .cache()
)
l2g_evidence_VEP.count()


5441

In [ ]:
l2g_evidence_eQTL = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full_diseases.filter(f.col("eQTL_coloc") == 1),
    score_column="score",
    datasource_id="l2g_eQTL",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)


In [ ]:
l2g_evidence_eQTL = (
    l2g_evidence_eQTL.select(columns_to_select)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("gwas_eQTL"))
    .cache()
)
l2g_evidence_eQTL = l2g_evidence_eQTL.join(
    l2g_evidence_VEP.select(columns_to_select), on=["targetId", "diseaseId"], how="left_anti"
).cache()
l2g_evidence_eQTL.count()


12343

In [ ]:
l2g_evidence_eQTL.columns


['targetId', 'diseaseId', 'resourceScore', 'source']

In [ ]:
qm_cs = (
    session.spark.read.parquet(path_to_intermediate_data_folder + "qualifying_measurement_credible_sets")
    .select("studyLocusId")
    .cache()
)
qm_cs.count()


450357

In [ ]:
l2g_full_all = session.spark.read.parquet(
    path_to_intermediate_data_folder + "list_of_prioritised_genes_per_CS_with_year_nfe_maf.parquet"
)


In [ ]:
l2g_measurements_full = l2g_full_all.join(qm_cs, on="studyLocusId", how="inner").drop("diseaseIds").cache()
l2g_measurements_full.count()


453009

In [ ]:
l2g_evidence_measurements = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_measurements_full,
    score_column="score",
    datasource_id="l2g_measurements",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)


In [ ]:
l2g_evidence_measurements = (
    l2g_evidence_measurements.select(columns_to_select)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("all_measurements"))
    .cache()
)
l2g_evidence_measurements.count()


150360

In [ ]:
l2g_evidence_measurements.select("targetId").distinct().count()


15160

In [ ]:
l2g_evidence_diseases.select("targetId").distinct().count()


8285

# OMIM


In [ ]:
evidence = session.spark.read.parquet(path_to_release_folder + "/output/evidence")


In [ ]:
evidence = (
    evidence.filter(f.col("score") >= 0)
    .filter(f.col("datasourceId").isin(["uniprot_variants", "uniprot_literature"]))
    .cache()
)


In [ ]:
evidence.groupBy("datasourceId").count().show()


+------------------+-----+
|      datasourceId|count|
+------------------+-----+
|  uniprot_variants|33047|
|uniprot_literature| 6744|
+------------------+-----+



In [ ]:
evidence_omim = evidence.filter(f.col("diseaseFromSourceId").substr(1, 4) == "OMIM").cache()
evidence_omim.count()


39791

In [ ]:
evidence_omim = (
    evidence_omim.select(columns_to_select)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("omim"))
    .cache()
)
evidence_omim.count()


6645

In [ ]:
enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence_omim,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0,
    efo_ancestors_to_remove=["MONDO_0045024"],
)
enrich


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,Relative success,ci_rs_low,ci_rs_high,rs_p_value,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.219015,2.191984e-01,0.900268,1.650617,1.030575,0.987809,1.075193,1.636971e-01,6114,30912,49,302,61009
1,3+,2.286880,4.792445e-14,1.834842,2.850283,1.450958,1.342529,1.568143,5.818247e-21,20450,16576,123,228,61009
2,4+,5.131773,1.406205e-42,4.139005,6.362663,3.436686,3.022431,3.907720,3.705331e-79,32606,4420,207,144,61009


# Orphanet evidence


In [ ]:
evidence = session.spark.read.parquet(path_to_release_folder + "/output/evidence")


In [ ]:
# evidence = evidence.filter(f.col("score")>=0.95).filter(f.col("datasourceId").isin(["eva", "uniprot_variants", "gene2phenotype", "genomics_england", "clingen","orphanet"])).cache()
evidence = evidence.filter(f.col("score") >= 0).filter(f.col("datasourceId").isin(["orphanet"])).cache()
evidence.count()


6301

In [ ]:
evidence_orphanet = (
    evidence.select(columns_to_select)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("orphanet"))
    .cache()
)
evidence_orphanet.count()


6288

In [ ]:
enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence_orphanet,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0,
    efo_ancestors_to_remove=["MONDO_0045024"],
)
enrich


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,Relative success,ci_rs_low,ci_rs_high,rs_p_value,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.194982,2.925021e-01,0.873670,1.634463,1.027683,0.982929,1.074473,2.293441e-01,6117,30936,46,278,59325
1,3+,2.366640,5.859771e-14,1.879705,2.979714,1.468201,1.356113,1.589553,2.586633e-21,20462,16591,111,213,59325
2,4+,4.676656,1.306846e-34,3.732320,5.859924,3.246845,2.824754,3.732008,1.058526e-61,32615,4438,198,126,59325


In [ ]:
evidence_orphanet.columns


['targetId', 'diseaseId', 'resourceScore', 'source']

# Genebased results


In [ ]:
evidence = session.spark.read.parquet(path_to_release_folder + "/output/evidence")


In [ ]:
# evidence.groupBy("datasourceId").count().show()


In [ ]:
evidence_gene_burden = evidence.filter(f.col("datasourceId") == "gene_burden").cache()
evidence_gene_burden.show(1)


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------------------+----------------+---------------+----------+--------+-------------------+-------------------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+--------------------+----------+------------+-----------+--------------+-------------+----+------------------------+-------------

In [ ]:
evidence_gene_burden.groupBy("statisticalMethod").count().show(100, truncate=False)


+----------------------------------------+-----+
|statisticalMethod                       |count|
+----------------------------------------+-----+
|ptvraredmg                              |3012 |
|ADD-WGR-FIRTH_M3.001                    |906  |
|pLoF                                    |2061 |
|ADD-WGR-FIRTH_M1.001                    |753  |
|LOF + missense0.8 (MAF<0.1%)            |293  |
|ADD-WGR-FIRTH_M1.1                      |953  |
|UR                                      |1521 |
|ADD-WGR-FIRTH_M3.01                     |1087 |
|LOF + missense0.5 (MAF<0.001%)          |190  |
|pLoF|missense|LC                        |2601 |
|ADD-WGR-FIRTH_M1.0001                   |482  |
|raredmg                                 |1155 |
|LOF (MAF<0.1%)                          |282  |
|URmtr                                   |1425 |
|ADD-WGR-FIRTH_M3.0001                   |517  |
|ADD-WGR-FIRTH_M1.01                     |842  |
|Cauchy                                  |403  |
|ADD-WGR-FIRTH_M3.1 

In [ ]:
evidence_gene_burden_reg = evidence_gene_burden.filter(f.array_contains(f.col("literature"), "34662886"))
evidence_gene_burden_reg.show(1)


+------------+---------------+-------------+-------------------+--------+--------------+-----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+---------------+----------------+---------------+----------+--------+-------------------+-------------------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+--------------------+---

In [ ]:
evidence_gene_burden_reg.groupBy("statisticalMethod").count().show(100, truncate=False)


+--------------------------+-----+
|statisticalMethod         |count|
+--------------------------+-----+
|ADD-WGR-FIRTH_M3.001      |906  |
|ADD-WGR-FIRTH_M1.001      |753  |
|ADD-WGR-FIRTH_M1.1        |953  |
|ADD-WGR-FIRTH_M3.01       |1087 |
|ADD-WGR-FIRTH_M1.0001     |482  |
|ADD-WGR-FIRTH_M3.0001     |517  |
|ADD-WGR-FIRTH_M1.01       |842  |
|ADD-WGR-FIRTH_M3.1        |1470 |
|ADD-WGR-FIRTH_M3.singleton|192  |
|ADD-WGR-FIRTH_M1.singleton|182  |
+--------------------------+-----+



In [ ]:
evidence.filter(f.col("datasourceId") == "gene_burden").groupBy("projectId").count().show(100, truncate=False)


+----------------------------+-----+
|projectId                   |count|
+----------------------------+-----+
|FinnGen                     |433  |
|AstraZeneca PheWAS Portal   |21296|
|NULL                        |196  |
|Genebass                    |7342 |
|SCHEMA consortium           |10   |
|REGENERON                   |7384 |
|Autism Sequencing Consortium|102  |
|CVDI Human Disease Portal   |1428 |
|OTAR022                     |21   |
|AMP-PD                      |9    |
|Epi25 collaborative         |2    |
+----------------------------+-----+



In [ ]:
evidence_genebass = evidence_gene_burden.filter(f.col("projectId") == "Genebass").filter(
    ~(f.col("statisticalMethod") == "synonymous")
)


In [ ]:
evidence_genebass.groupBy("statisticalMethod").count().show(100, truncate=False)


+-----------------+-----+
|statisticalMethod|count|
+-----------------+-----+
|pLoF             |2061 |
|pLoF|missense|LC |2601 |
|missense|LC      |2127 |
+-----------------+-----+



In [ ]:
evidence_finngen = evidence_gene_burden.filter(f.col("projectId") == "FinnGen")


In [ ]:
evidence_AstraZeneca = evidence_gene_burden.filter(f.col("projectId") == "AstraZeneca PheWAS Portal")
evidence_AstraZeneca.groupBy("statisticalMethod").count().show(100, truncate=False)


+-----------------+-----+
|statisticalMethod|count|
+-----------------+-----+
|ptvraredmg       |3012 |
|UR               |1521 |
|raredmg          |1155 |
|URmtr            |1425 |
|ptv              |3392 |
|rec              |271  |
|syn              |56   |
|raredmgmtr       |565  |
|ptv5pcnt         |4346 |
|flexdmg          |3398 |
|flexnonsynmtr    |2155 |
+-----------------+-----+



In [ ]:
evidence_AstraZeneca = evidence_AstraZeneca.filter(~(f.col("statisticalMethod") == "syn"))


In [ ]:
evidence_genebass = evidence_genebass.select(columns_to_select).distinct()
evidence_gene_burden_reg = evidence_gene_burden_reg.select(columns_to_select).distinct()
evidence_finngen = evidence_finngen.select(columns_to_select).distinct()
evidence_AstraZeneca = evidence_AstraZeneca.select(columns_to_select).distinct()


In [ ]:
evidence_gene_burden = (
    evidence_genebass.unionByName(evidence_gene_burden_reg)
    .unionByName(evidence_finngen)
    .unionByName(evidence_AstraZeneca)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("gene_burden"))
    .cache()
)
evidence_gene_burden.count()


6884

In [ ]:
evidence_gene_burden.show(1)


+---------------+-----------+-------------+-----------+
|       targetId|  diseaseId|resourceScore|     source|
+---------------+-----------+-------------+-----------+
|ENSG00000089820|EFO_0004587|          1.0|gene_burden|
+---------------+-----------+-------------+-----------+
only showing top 1 row



In [ ]:
evidence_gene_burden.select("targetId").distinct().count()


1699

In [ ]:
evidence_gene_burden.show(1)


+---------------+-----------+-------------+-----------+
|       targetId|  diseaseId|resourceScore|     source|
+---------------+-----------+-------------+-----------+
|ENSG00000089820|EFO_0004587|          1.0|gene_burden|
+---------------+-----------+-------------+-----------+
only showing top 1 row



In [ ]:
enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence_gene_burden,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0,
    efo_ancestors_to_remove=["MONDO_0045024"],
)
enrich


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,Relative success,ci_rs_low,ci_rs_high,rs_p_value,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.876780,2.981891e-01,0.669587,5.260409,1.083503,0.982134,1.195334,1.095344e-01,6159,31176,4,38,15428
1,3+,3.923301,6.242766e-05,1.928154,7.982914,1.696024,1.431679,2.009177,9.900233e-10,20563,16772,10,32,15428
2,4+,7.218138,2.586892e-09,3.939022,13.227017,4.109069,3.032952,5.567001,7.402276e-20,32792,4543,21,21,15428


# ChEMBL


In [ ]:
platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)


In [ ]:
chembl_evidence_clean = chembl_evidence.filter(f.col("clinicalPhase") >= 3)
chembl_evidence_clean = (
    chembl_evidence_clean.select(columns_to_select)
    .distinct()
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("ChEMBL"))
)
chembl_evidence_clean.count()


25685

In [ ]:
chembl_evidence_clean.select("targetId").distinct().count()


1146

In [ ]:
chembl_evidence_clean.show(1)


+---------------+-------------+-------------+------+
|       targetId|    diseaseId|resourceScore|source|
+---------------+-------------+-------------+------+
|ENSG00000004779|MONDO_0020121|          1.0|ChEMBL|
+---------------+-------------+-------------+------+
only showing top 1 row



In [ ]:
efo_cancer = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig, efo_ids=["MONDO_0045024"]
)


In [ ]:
len(efo_cancer)


3593

In [ ]:
chembl_cancer = (
    chembl_evidence_clean.filter(f.col("diseaseId").isin(efo_cancer))
    .withColumn("source", f.lit("cancer_ChEMBL"))
    .cache()
)
chembl_cancer.count()


8881

In [ ]:
chembl_non_cancer = (
    chembl_evidence_clean.filter(~(f.col("diseaseId").isin(efo_cancer)))
    .withColumn("source", f.lit("non_cancer_ChEMBL"))
    .cache()
)
chembl_non_cancer.count()


16804

In [ ]:
chembl_evidence_clean.show(1)


+---------------+-------------+-------------+------+
|       targetId|    diseaseId|resourceScore|source|
+---------------+-------------+-------------+------+
|ENSG00000004779|MONDO_0020121|          1.0|ChEMBL|
+---------------+-------------+-------------+------+
only showing top 1 row



In [ ]:
chembl_cancer.show(1)


+---------------+-----------+-------------+-------------+
|       targetId|  diseaseId|resourceScore|       source|
+---------------+-----------+-------------+-------------+
|ENSG00000007314|EFO_1000249|          1.0|cancer_ChEMBL|
+---------------+-----------+-------------+-------------+
only showing top 1 row



In [ ]:
chembl_non_cancer.show(1)


+---------------+-----------+-------------+-----------------+
|       targetId|  diseaseId|resourceScore|           source|
+---------------+-----------+-------------+-----------------+
|ENSG00000007314|EFO_0004699|          1.0|non_cancer_ChEMBL|
+---------------+-----------+-------------+-----------------+
only showing top 1 row



# molQTLs


In [ ]:
sl.df.groupby("studyType").count().show()


+---------+-------+
|studyType|  count|
+---------+-------+
|     gwas| 789453|
|     sqtl| 223500|
|     pqtl|  33731|
|    tuqtl| 384852|
|     eqtl|1349478|
|   sceqtl|  52744|
+---------+-------+



In [ ]:
molQTL_evidence = (
    sl.df.filter(~(f.col("studyType") == "gwas"))
    .select("studyId")
    .join(si.df.select("studyId", "geneId"), "studyId", "inner")
    .cache()
)
molQTL_evidence.count()


2044305

In [ ]:
molQTL_evidence.show(1)


+--------------------+---------------+
|             studyId|         geneId|
+--------------------+---------------+
|gtex_tx_colon_tra...|ENSG00000133030|
+--------------------+---------------+
only showing top 1 row



In [ ]:
molQTL_evidence = molQTL_evidence.select("geneId").distinct().withColumnRenamed("geneId", "targetId").cache()
molQTL_evidence.count()


29342

In [ ]:
molQTL_evidence = (
    molQTL_evidence.withColumn("diseaseId", f.lit("molQTL"))
    .withColumn("resourceScore", f.lit(1.0))
    .withColumn("source", f.lit("molQTL"))
    .cache()
)
molQTL_evidence.count()


29342

# Load target


In [ ]:
target = session.spark.read.parquet(path_to_release_folder + "/output/target")


In [ ]:
# Create a list of valid chromosomes
valid_chromosomes = [str(i) for i in range(1, 23)] + ["X", "Y"]


In [ ]:
target.groupBy("genomicLocation.chromosome").count().show()


+--------------------+-----+
|          chromosome|count|
+--------------------+-----+
|                   7| 3957|
|                  15| 2819|
|                  11| 4166|
|                   3| 4158|
|                   8| 3266|
|                  22| 1747|
|                  16| 3187|
|HSCHR19KIR_FH05_B...|    2|
|                   5| 3945|
|                  18| 1654|
|                   Y|  672|
|                  17| 3778|
|                  MT|   37|
|                   6| 4230|
|                  19| 3547|
|                   X| 2955|
|                   9| 3188|
|                   1| 7089|
|                  20| 1969|
|                  10| 3268|
+--------------------+-----+
only showing top 20 rows



In [ ]:
# target=session.spark.read.parquet("gs://open-targets-data-releases/25.06/output/target")
target = (
    target.filter(f.col("genomicLocation").getField("chromosome").isin(valid_chromosomes))
    .filter(f.col("biotype") == "protein_coding")
    .cache()
)
target.count()


20083

In [ ]:
target = target.withColumnRenamed("id", "targetId").cache()
target.show(1)


+---------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+----------------+-------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+----------------+--------------------+--------------------+----+--------------------+--------------------+--------------+--------------------+--------------------+-----------------+--------+---------+
|       targetId|approvedSymbol|       biotype|       transcriptIds| canonicalTranscript|      canonicalExons|     genomicLocation|alternativeGenes| approvedName|                  go|hallmarks|            synonyms|      symbolSynonyms|        nameSynonyms|functionDescriptions|subcellularLocations|targetClass| obsoleteSymbols|       obsoleteNames|          constraint| tep|          proteinIds|             dbXrefs|chemicalProbes|          homologues|        tractability|safetyLiabilitie

In [ ]:
target.printSchema()


root
 |-- targetId: string (nullable = true)
 |-- approvedSymbol: string (nullable = true)
 |-- biotype: string (nullable = true)
 |-- transcriptIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- canonicalTranscript: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: string (nullable = true)
 |-- canonicalExons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genomicLocation: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: integer (nullable = true)
 |-- alternativeGenes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- approvedName: string (nullable = true)
 |-- go: array (nullable = true)
 |    |-- element: struct (containsNull

In [ ]:
target.show(1)


+---------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+----------------+-------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+----------------+--------------------+--------------------+----+--------------------+--------------------+--------------+--------------------+--------------------+-----------------+--------+---------+
|       targetId|approvedSymbol|       biotype|       transcriptIds| canonicalTranscript|      canonicalExons|     genomicLocation|alternativeGenes| approvedName|                  go|hallmarks|            synonyms|      symbolSynonyms|        nameSynonyms|functionDescriptions|subcellularLocations|targetClass| obsoleteSymbols|       obsoleteNames|          constraint| tep|          proteinIds|             dbXrefs|chemicalProbes|          homologues|        tractability|safetyLiabilitie

# Combine


In [ ]:
efo_measurmnets = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig, efo_ids=["EFO_0001444"]
)


In [ ]:
len(efo_measurmnets)


18668

In [ ]:
combined_evidence_protein_coding_genes = (
    evidence_orphanet.unionByName(evidence_omim)
    .unionByName(l2g_evidence_diseases)
    .unionByName(evidence_gene_burden)
    .unionByName(chembl_evidence_clean)
    .unionByName(l2g_evidence_measurements)
    .unionByName(l2g_evidence_eQTL)
    .unionByName(l2g_evidence_VEP)
    .unionByName(molQTL_evidence)
    .unionByName(chembl_cancer)
    .unionByName(chembl_non_cancer)
    .join(target.select("targetId"), on="targetId", how="inner")
    .cache()
)
combined_evidence_protein_coding_genes.count()


26/01/21 14:19:24 WARN DAGScheduler: Broadcasting large task binary with size 1047.4 KiB
26/01/21 14:19:41 WARN DAGScheduler: Broadcasting large task binary with size 1051.6 KiB


292870

In [ ]:
combined_evidence_protein_coding_genes.filter(f.col("targetId").isNull()).count()


26/01/21 14:20:20 WARN DAGScheduler: Broadcasting large task binary with size 1052.3 KiB


0

In [ ]:
combined_evidence_protein_coding_genes.filter(f.col("diseaseId").isNull()).count()


26/01/21 14:20:50 WARN DAGScheduler: Broadcasting large task binary with size 1052.3 KiB


0

In [ ]:
combined_evidence_protein_coding_genes.select("targetId").distinct().count()


26/01/21 14:23:52 WARN DAGScheduler: Broadcasting large task binary with size 1060.3 KiB
26/01/21 14:24:45 WARN DAGScheduler: Broadcasting large task binary with size 1068.0 KiB


18809

In [ ]:
combined_evidence_protein_coding_genes.toPandas().to_csv(
    path_to_intermediate_data_folder + "combined_evidence_with_measurements.csv", index=False
)


26/01/21 14:25:01 WARN DAGScheduler: Broadcasting large task binary with size 1048.7 KiB


In [ ]:
target.select("targetId").distinct().count()


20083

In [ ]:
combined_evidence_protein_coding_genes.show(1)


+---------------+-------------+-------------+--------+
|       targetId|    diseaseId|resourceScore|  source|
+---------------+-------------+-------------+--------+
|ENSG00000064419|MONDO_0012034|          1.0|orphanet|
+---------------+-------------+-------------+--------+
only showing top 1 row



26/01/21 14:25:34 WARN DAGScheduler: Broadcasting large task binary with size 1051.4 KiB


In [ ]:
combined_evidence_protein_coding_genes.groupBy("source").count().show()


26/01/21 14:25:38 WARN DAGScheduler: Broadcasting large task binary with size 1061.6 KiB
26/01/21 14:26:04 WARN DAGScheduler: Broadcasting large task binary with size 1061.6 KiB


+-----------------+------+
|           source| count|
+-----------------+------+
|         orphanet|  6192|
|             omim|  6596|
|     all_diseases| 36858|
|      gene_burden|  6857|
|           ChEMBL| 25234|
| all_measurements|150360|
|        gwas_eQTL| 12343|
|    gwas_with_pav|  5441|
|           molQTL| 17755|
|    cancer_ChEMBL|  8811|
|non_cancer_ChEMBL| 16423|
+-----------------+------+



In [ ]:
combined_evidence_protein_coding_genes_diseases = (
    combined_evidence_protein_coding_genes.filter(~(f.col("source") == "molQTL"))
    .filter(~(f.col("source") == "all_measurements"))
    .filter(~(f.col("diseaseId").isin(efo_measurmnets)))
    .cache()
)


In [ ]:
combined_evidence_protein_coding_genes_diseases.count()


26/01/21 14:26:31 WARN DAGScheduler: Broadcasting large task binary with size 1491.4 KiB
26/01/21 14:27:08 WARN DAGScheduler: Broadcasting large task binary with size 1495.6 KiB


119665

In [ ]:
combined_evidence_protein_coding_genes_diseases.select("targetId").distinct().count()


26/01/21 14:27:47 WARN DAGScheduler: Broadcasting large task binary with size 1504.4 KiB
26/01/21 14:28:36 WARN DAGScheduler: Broadcasting large task binary with size 1512.0 KiB


11022

In [ ]:
combined_evidence_protein_coding_genes_diseases.groupBy("source").count().show()


26/01/21 14:28:46 WARN DAGScheduler: Broadcasting large task binary with size 1505.6 KiB
26/01/21 14:29:28 WARN DAGScheduler: Broadcasting large task binary with size 1505.7 KiB


+-----------------+-----+
|           source|count|
+-----------------+-----+
|         orphanet| 6192|
|             omim| 6596|
|     all_diseases|36858|
|      gene_burden| 1777|
|           ChEMBL|25229|
|        gwas_eQTL|12343|
|    gwas_with_pav| 5441|
|    cancer_ChEMBL| 8811|
|non_cancer_ChEMBL|16418|
+-----------------+-----+



In [ ]:
pleiotropy_combined_evidence = (
    combined_evidence_protein_coding_genes_diseases.groupBy("targetId")
    .agg(f.countDistinct("diseaseId").alias("unique_disease_count"))
    .cache()
)
pleiotropy_combined_evidence.count()


26/01/21 14:29:29 WARN DAGScheduler: Broadcasting large task binary with size 1505.6 KiB
26/01/21 14:30:06 WARN DAGScheduler: Broadcasting large task binary with size 1521.5 KiB
26/01/21 14:30:06 WARN DAGScheduler: Broadcasting large task binary with size 1515.9 KiB
26/01/21 14:30:10 WARN DAGScheduler: Broadcasting large task binary with size 1520.2 KiB


11022

In [ ]:
pleiotropy_combined_evidence.show(1)


+---------------+--------------------+
|       targetId|unique_disease_count|
+---------------+--------------------+
|ENSG00000070182|                   7|
+---------------+--------------------+
only showing top 1 row



26/01/21 14:30:15 WARN DAGScheduler: Broadcasting large task binary with size 1519.3 KiB


In [ ]:
pleiotropy_combined_evidence.toPandas().to_csv(
    path_to_intermediate_data_folder + "pleiotropy_combined_evidence.csv", index=False
)


26/01/21 14:31:33 WARN DAGScheduler: Broadcasting large task binary with size 1517.1 KiB


In [ ]:
combined_evidence_protein_coding_genes_diseases.toPandas().to_csv(
    path_to_intermediate_data_folder + "combined_evidence_without_measurements.csv", index=False
)


26/01/21 14:31:41 WARN DAGScheduler: Broadcasting large task binary with size 1492.7 KiB


# Combine with target


In [ ]:
target = target.select("targetId", "biotype", "constraint")
target.show(5, truncate=False)


+---------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|targetId       |biotype       |constraint                                                                                                                                                                                               |
+---------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ENSG00000000003|protein_coding|[{syn, -0.47468, 30.657, 34, 1.1091, 0.843, 1.476, NULL, NULL, NULL}, {mis, 0.59223, 80.999, 66, 0.81482, 0.667, 1.0, NULL, NULL, NULL}, {lof, 0.066079, 7.865, 3, 0.38144, 0.173, 0.985, 10507, 5, 3}]  |
|ENSG00000000005|protein_coding|[{syn, -0.46371, 38.347, 42,

In [ ]:
from pyspark.sql.functions import col, element_at, expr, size, when

target_with_constraints = (
    target.withColumn("syn_constr", expr("filter(constraint, x -> x.constraintType = 'syn')[0].score"))
    .withColumn("mis_constr", expr("filter(constraint, x -> x.constraintType = 'mis')[0].score"))
    .withColumn("lof_constr", expr("filter(constraint, x -> x.constraintType = 'lof')[0].oeUpper"))
)


In [ ]:
target_with_constraints.show(1)


+---------------+--------------+--------------------+----------+----------+----------+
|       targetId|       biotype|          constraint|syn_constr|mis_constr|lof_constr|
+---------------+--------------+--------------------+----------+----------+----------+
|ENSG00000000003|protein_coding|[{syn, -0.47468, ...|  -0.47468|   0.59223|     0.985|
+---------------+--------------+--------------------+----------+----------+----------+
only showing top 1 row



In [ ]:
target_with_constraints = target_with_constraints.drop("constraint").toPandas()


In [ ]:
target_with_constraints


,targetId,biotype,syn_constr,mis_constr,lof_constr
0,ENSG00000000003,protein_coding,-0.47468,0.59223,0.985
1,ENSG00000000005,protein_coding,-0.46371,0.85895,0.910
2,ENSG00000001084,protein_coding,0.58706,2.54940,0.401
3,ENSG00000003137,protein_coding,-0.66253,1.04920,0.290
4,ENSG00000004059,protein_coding,0.64570,2.27720,0.330
...,...,...,...,...,...
20078,ENSG00000292348,protein_coding,NaN,NaN,NaN
20079,ENSG00000293560,protein_coding,NaN,NaN,NaN
20080,ENSG00000293600,protein_coding,NaN,NaN,NaN
20081,ENSG00000293663,protein_coding,NaN,NaN,NaN


In [ ]:
target_with_constraints["lof_constr"].describe()


count    18322.000000
mean         0.923225
std          0.515831
min          0.030000
25%          0.483000
50%          0.880500
75%          1.301000
max          1.996000
Name: lof_constr, dtype: float64

In [ ]:
## Check before and after
# print("Before:")
# print(f"NaN count: {target_with_constraints['lof_constr'].isnull().sum()}")
# print(f"Mean: {target_with_constraints['lof_constr'].mean():.4f}")

# Replace NaN with mean
# target_with_constraints['lof_constr'] = target_with_constraints['lof_constr'].fillna(target_with_constraints['lof_constr'].mean())

# print("\nAfter:")
# print(f"NaN count: {target_with_constraints['lof_constr'].isnull().sum()}")
# print(f"Mean: {target_with_constraints['lof_constr'].mean():.4f}")


In [ ]:
target_with_constraints["lof_constr"].describe()


count    18322.000000
mean         0.923225
std          0.515831
min          0.030000
25%          0.483000
50%          0.880500
75%          1.301000
max          1.996000
Name: lof_constr, dtype: float64

In [ ]:
target_with_constraints["lof_constr"] = -target_with_constraints["lof_constr"]


In [ ]:
target_with_constraints["lof_constr"].describe()


count    18322.000000
mean        -0.923225
std          0.515831
min         -1.996000
25%         -1.301000
50%         -0.880500
75%         -0.483000
max         -0.030000
Name: lof_constr, dtype: float64

In [ ]:
target_with_constraints.to_csv(path_to_intermediate_data_folder + "target_with_constraints_2509.csv", index=False)
